In [1]:
# Import required libraries
import pandas as pd
import numpy as np 
import math 
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import os
import glob
import re
from PIL import Image
from pathlib import Path
import base64, io

In [ ]:
# 0) OPTIONAL  ▸ set to 1031  or  [1031, 1142] (Participant Identifier)  to load only these, if it should load all Participants: set it to None
participants_to_load = None

# 1) WHAT TO LOAD – just edit this list
sessions_to_load = [1]          # e.g. [1, 2, 3...] to load several sessions

# 2) WHERE YOUR RAW CSVs LIVE
folder = r"C:/Users/Marcel/Desktop/Study Project/VR Data/Data Classified"

# 3) WHERE TO SAVE THE PROCESSED FILES (created if absent)
base_folder = os.path.dirname(folder)         
out_folder  = os.path.join(base_folder, "Data with Turns")
os.makedirs(out_folder, exist_ok=True)

# LOAD ONLY THE DESIRED SESSION FILES
pattern = re.compile(r'(\d{4})_(\d).*\.csv$')                 # matches the 1234_1.csv format
dfs = []  # will collect DataFrames here

for filepath in glob.glob(os.path.join(folder, "*.csv")):
    m = pattern.search(os.path.basename(filepath))
    if not m:
        continue
    pid, sess = map(int, m.groups())

    # --- participant filter -------------------------------------------
    if participants_to_load is not None:
        # allow either a single int or an iterable
        if isinstance(participants_to_load, (list, tuple, set)):
            if pid not in participants_to_load:
                continue
        else:              # single int
            if pid != participants_to_load:
                continue

    # --- session filter -----------------------------------------------
    if sess not in sessions_to_load:
        continue

    # --- load ----------------------------------------------------------
    df = pd.read_csv(filepath, sep=",")
    dfs.append(df)

# SUMMARY OF WHAT GOT LOADED
print(f"Loaded {len(dfs)} file(s) for session(s) {sessions_to_load} "
      f"{'and participants ' + str(participants_to_load) if participants_to_load else '(all participants)'}."
)

Loaded 1 file(s) for session(s) [1] and participants 1031.


In [4]:
## Plotting function with City Map Image roughly aligned
# ── 1)  load the PNG once at import time --------------------------------
MAP_PATH = Path(r"C:/Users/Marcel/Desktop/Study Project/VR Data/city_map.png")
IMG      = Image.open(MAP_PATH)          # PIL image
IMG_W, IMG_H = IMG.size                  # 787 × 632  (pixels)

# ── 2)  fixed alignment constants ------------------
OFFSET_X = -490      
OFFSET_Y =  400      
SCALE    = 0.79      

def plot_path(
    df,
    *,
    highlight=None,                     # iterable of indices → red “X”
    title="Path with map overlay",
    width=1400, height=1000,
    opacity=1.0
):
    """
    Draw the participant trajectory with the fixed city_map PNG beneath.
    """

    # --- scatter layer -------------------------------------------------
    fig = px.scatter(
        df,
        x="playerBodyPosition.x",
        y="playerBodyPosition.z",
        color=df.index,
        hover_data={"total_time": ':.2f'},
        color_continuous_scale="Viridis",
        labels={"playerBodyPosition.x": "Body X",
                "playerBodyPosition.z": "Body Z",
                "total_time": "Time",
                "color": "Index"},
        title=title,
        width=width, height=height
    )
    fig.update_layout(
        yaxis=dict(scaleanchor="x", scaleratio=1),
        dragmode="pan", margin=dict(l=0, r=0, t=50, b=0)
    )

    # --- drop the map behind the scatter ------------------------------
    fig.add_layout_image(
        dict(
            source=IMG,                     
            xref="x", yref="y",
            x=OFFSET_X,                     
            y=OFFSET_Y,
            sizex=IMG_W / SCALE,            
            sizey=IMG_H / SCALE,            
            sizing="stretch",
            layer="below",
            opacity=opacity
        )
    )

    # --- keep axes exactly the map rectangle --------------------------
    fig.update_xaxes(range=[OFFSET_X, OFFSET_X + IMG_W / SCALE])
    fig.update_yaxes(range=[OFFSET_Y, OFFSET_Y + IMG_H / SCALE])

    # --- optional highlight points ------------------------------------
    if highlight is not None and len(highlight):
        sel = df.loc[highlight]

        # put BOTH the row‑index and total_time into customdata
        cd = np.column_stack([sel.index, sel["total_time"]])

        fig.add_trace(
            go.Scatter(
                x=sel["playerBodyPosition.x"],
                y=sel["playerBodyPosition.z"],
                mode="markers",
                marker=dict(color="red", size=12, symbol="circle"),
                name="highlight",
                customdata=cd,                     
                hovertemplate=(
                    "index %{customdata[0]}<br>" 
                    "time %{customdata[1]:.2f}s<br>"             
                    "X =%{x:.2f}<br>Z =%{y:.2f}"           
                    "<extra></extra>"                      
                )
            )
        )

    # --- show ----------------------------------------------------------
    fig.show(renderer="browser",
             config={"responsive": True, "scrollZoom": True})

In [5]:
# Plotting Path Trajectory with the plotly package as interactive plot for each participant
for df in dfs:
    # --- 1) initial plot (no highlights) ------------------------------
    pid, sess = df["SubjectID"].iat[0], df["Session"].iat[0]
    title = f"Participant {pid} / Session {sess}"
    plot_path(df, title=title)
    
    # pause so you can inspect & type indices
    # ---------- FIRST‑VISIT indices ---------------------------------------
    turn_input = input(
        f"Enter comma-separated FIRST-visit indices for {pid}_{sess} "
        "(leave blank to skip this participant) » "
    ).strip()
    if not turn_input:
        print(f"  ↪ skipped participant {pid} / session {sess}\n")
        continue  
    
    first_idxs = [int(x) for x in turn_input.split(",") if x.strip().isdigit()]

    # assign entry_nr = 1 to all first-visit indices
    df["entry_nr"] = np.nan
    df.loc[df.index.isin(first_idxs), "entry_nr"] = 1

    # initialise columns (everything else NaN/False)
    plot_path(df,
              highlight=first_idxs,
              title=f"{pid}_{sess} — first visits highlighted")
              
    df["street_id_within_participant"] = np.nan

    # ---------- repeated‑street PAIRS -------------------------------------
    while True:
        pair_str = input(
        "Repeated-street groups  "
        "(examples: 4500:89161  or  4500:89161:94001, 56566:85291) "
        "[Enter = none] » "
        ).strip()

        if not pair_str:                      # user pressed Enter → no pairs
            print("  ↪ no pairs entered — skipping this participant\n")                  
            break            # exit the while‑loop 

        bad   = []                      # collect all issues this round
        groups = []                      # valid (first, second) tuples

        for token in pair_str.split(","):
            token = token.strip()
            if not token:
                continue

            idx_strings = token.split(":")
            if len(idx_strings) < 2:
                bad.append(f"need ≥2 indices ⇒ “{token}”")
                continue
            if not all(s.strip().isdigit() for s in idx_strings):
                bad.append(f"non-int value ⇒ “{token}”")
                continue

            indices = [int(s) for s in idx_strings]

            # 1) every index must exist
            missing = [i for i in indices if i not in df.index]
            if missing:
                bad.append(f"indices not found {missing} in “{token}”")
                continue

            # 2) first index of the group must be in first‑visit list
            if indices[0] not in first_idxs:
                bad.append(f"first idx {indices[0]} not in FIRST-visit list")
                continue

            groups.append(indices)

        # any problems?  → show them & re‑prompt
        if bad:
            print("\nIssues detected:")
            for msg in bad:
                print("  •", msg)
            print("Please re-enter the entire pair list.\n")
            continue       # back to while‑loop prompt

        # ---------------- write the valid pairs ---------------------------
        sid_counter = 1
        for grp in groups:
            for entry_nr, idx in enumerate(grp, start=1):
                df.at[idx, "street_id_within_participant"] = sid_counter
                # only assign entry_nr if not already 1 (first visits already labeled)
                if pd.isna(df.at[idx, "entry_nr"]):
                    df.at[idx, "entry_nr"] = entry_nr
            sid_counter += 1
        break       # all good → exit the while‑loop
     
    # save
    out_name = f"{pid:04d}_{sess}_wTurns.csv"
    df.to_csv(os.path.join(out_folder, out_name), index=False)
    print(f"Processed & saved {out_name}")

Processed & saved 1031_1_wTurns.csv
